## Imports


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt

print("TF version:", tf.__version__)


## Data: placeholder synthetic example
    Replace this with your real data loader.


In [ ]:
# sampling time
dt = 0.001  # 1 ms
T = 20000   # samples
t = np.arange(T) * dt

# ground-truth (unknown in practice)
true_J  = 0.02
true_B  = 0.05
true_Kt = 0.8

# synthetic torque-producing current
iq = 2.0 + 0.5*np.sin(2*np.pi*2*t) + 0.3*np.sin(2*np.pi*10*t)

omega = np.zeros(T)
for k in range(1, T):
    domega = (true_Kt*iq[k-1] - true_B*omega[k-1]) / true_J
    omega[k] = omega[k-1] + dt*domega

# add measurement noise
omega_meas = omega + 0.02*np.random.randn(T)



## Sliding window builder


In [ ]:
def build_windows(iq, omega, window=400, step=50):
    """
    iq, omega: 1D arrays (T,)
    Returns tf.float32 arrays: (N, window, 1)
    """
    X, Y = [], []
    T = len(iq)
    for start in range(0, T-window, step):
        end = start + window
        X.append(iq[start:end])
        Y.append(omega[start:end])
    X = np.array(X).reshape(-1, window, 1).astype("float32")
    Y = np.array(Y).reshape(-1, window, 1).astype("float32")
    return X, Y

iq_win, om_win = build_windows(iq, omega_meas, window=400, step=50)

# train/val split
n = iq_win.shape[0]
n_train = int(0.8 * n)
iq_train, iq_val = iq_win[:n_train], iq_win[n_train:]
om_train, om_val = om_win[:n_train], om_win[n_train:]

train_ds = (tf.data.Dataset.from_tensor_slices((iq_train, om_train))
            .shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE))
val_ds   = (tf.data.Dataset.from_tensor_slices((iq_val, om_val))
            .batch(32).prefetch(tf.data.AUTOTUNE))



## Gray-box dynamic model
$$
\omega_{k+1} = \omega_k + dt * \frac{1}{J}*(Kt*iq_k - B*\omega_k - \tau_{res})
$$    


In [ ]:


class SpindleGrayBox(Model):
    def __init__(self, hidden_units=32, dt=0.001):
        super().__init__()
        self.dt = dt

        self.residual_net = tf.keras.Sequential([
            layers.Dense(hidden_units, activation="tanh"),
            layers.Dense(hidden_units, activation="tanh"),
            layers.Dense(1)
        ])

        def init_softplus_param(x0):
            return tf.Variable(
                tf.math.log(tf.exp(tf.constant(x0, tf.float32)) - 1.0),
                trainable=True
            )

        self.log_J  = init_softplus_param(0.01)
        self.log_B  = init_softplus_param(0.05)
        self.log_Kt = init_softplus_param(1.0)

    def call(self, iq_seq, omega0):
        """
        iq_seq:  (batch, T, 1)
        omega0:  (batch, 1)
        returns:
            omega_pred: (batch, T, 1)
        """
        J  = tf.nn.softplus(self.log_J)
        B  = tf.nn.softplus(self.log_B)
        Kt = tf.nn.softplus(self.log_Kt)
        dt = self.dt

        batch_size = tf.shape(iq_seq)[0]
        T = tf.shape(iq_seq)[1]

        omega_t = omega0  # (batch, 1)
        omega_pred = []

        for k in range(T):
            iq_k = iq_seq[:, k:k+1, :]          # (batch, 1, 1)
            x = tf.concat([omega_t, iq_k], -1)  # (batch, 1, 2)
            tau_res = self.residual_net(x)      # (batch, 1, 1)

            domega = (Kt * iq_k - B * omega_t - tau_res) / J
            omega_t = omega_t + dt*domega

            omega_pred.append(omega_t)

        return tf.concat(omega_pred, axis=1)

model = SpindleGrayBox(hidden_units=32, dt=dt)



## Training

In [ ]:
optimizer = tf.keras.optimizers.Adam(1e-3)
mse = tf.keras.losses.MeanSquaredError()

@tf.function
def train_step(iq_batch, om_batch):
    omega0 = om_batch[:, 0:1, :]  # initial state per window
    with tf.GradientTape() as tape:
        om_pred = model(iq_batch, omega0)
        loss = mse(om_batch, om_pred)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

@tf.function
def val_step(iq_batch, om_batch):
    omega0 = om_batch[:, 0:1, :]
    om_pred = model(iq_batch, omega0)
    return mse(om_batch, om_pred)

n_epochs = 40
for epoch in range(1, n_epochs+1):
    train_losses = []
    for iq_b, om_b in train_ds:
        L = train_step(iq_b, om_b)
        train_losses.append(L)
    train_loss = tf.reduce_mean(train_losses)

    val_losses = []
    for iq_b, om_b in val_ds:
        L = val_step(iq_b, om_b)
        val_losses.append(L)
    val_loss = tf.reduce_mean(val_losses)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:03d}: train={train_loss.numpy():.6e}, "
              f"val={val_loss.numpy():.6e}")

print("Estimated params:")
print("J_eff  =", tf.nn.softplus(model.log_J).numpy())
print("B_eff  =", tf.nn.softplus(model.log_B).numpy())
print("Kt_eff =", tf.nn.softplus(model.log_Kt).numpy())




## Plot prediction on one long segment


In [ ]:

# Take first validation window, roll forward
iq_sample = iq_val[0:1]   # (1, T, 1)
om_sample = om_val[0:1]   # (1, T, 1)
omega0    = om_sample[:, 0:1, :]

om_pred = model(iq_sample, omega0).numpy().squeeze()
om_true = om_sample.numpy().squeeze()

plt.figure(figsize=(10,4))
plt.plot(om_true, label="measured")
plt.plot(om_pred, "--", label="model")
plt.xlabel("sample (within window)")
plt.ylabel("omega")
plt.legend()
plt.tight_layout()
plt.show()
